# RAG Homework

This notebook implements a RAG architecture to answer questions about the Federal Law for the Prevention and Identification of Operations with Resources of Illicit Origin. The workflow includes data loading, preprocessing, chunking, retriever construction, answer generation, and automatic evaluation of multiple configurations.


## 1. Setup and imports

This section loads the required libraries, configures the local project path, and initializes the OpenAI client used for generation, embeddings, and evaluation. It also imports the `rag_utils.py` module, where the main project classes are encapsulated.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from openai import OpenAI
from rag_utils import RAG, EVALS

sys.path.append(str(Path.cwd().parent))
from openai_apikey import key

client = OpenAI(api_key=key)


## 2. Data loading

This section loads the legal document and defines the questions used throughout the experiment. The full text acts as the knowledge source for the RAG system.


### 2.1 Source document

Here, the plain text file of the law is loaded and the total number of lines and characters is checked to verify that the file was read correctly.


In [2]:
data_path = Path("Compilado_LFPIORPI20mayo2021.txt")
raw_text = data_path.read_text(encoding="utf-8")

lines = raw_text.splitlines()
print(f"Number of lines: {len(lines):,}")
print(f"Number of characters: {len(raw_text):,}")


Number of lines: 1,293
Number of characters: 106,311


### 2.2 Homework questions

Here, the homework questions are defined. These questions are used both to evaluate the RAG configurations and to generate the final answers submitted for the assignment.


In [3]:
homework_questions = [
    "¿Cuál es el objeto de la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita?",
    "¿Qué autoridades llevarán a cabo la supervisión, verificación y vigilancia del cumplimiento de las obligaciones de las Entidades Financieras?",
    "¿A partir de qué monto la comercialización o distribución de vehículos se considera una Actividad Vulnerable sujeta a Aviso?",
    "¿Cuál es el plazo para presentar un Aviso en caso de tener sospecha de que los recursos pudieran provenir de la comisión de delitos de operaciones con recursos de procedencia ilícita?",
    "¿Cuáles son las sanciones por proporcionar información o documentación falsa destinada a ser incorporada en los Avisos?",
]

for question_number, question in enumerate(homework_questions, start=1):
    print(f"{question_number}. {question}")


1. ¿Cuál es el objeto de la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita?
2. ¿Qué autoridades llevarán a cabo la supervisión, verificación y vigilancia del cumplimiento de las obligaciones de las Entidades Financieras?
3. ¿A partir de qué monto la comercialización o distribución de vehículos se considera una Actividad Vulnerable sujeta a Aviso?
4. ¿Cuál es el plazo para presentar un Aviso en caso de tener sospecha de que los recursos pudieran provenir de la comisión de delitos de operaciones con recursos de procedencia ilícita?
5. ¿Cuáles son las sanciones por proporcionar información o documentación falsa destinada a ser incorporada en los Avisos?


## 3. RAG Class showcase

This section demonstrates how the RAG class works step by step using a fixed configuration. The goal is to showcase the custom-made features developed for this legal-document problem: legal-format preprocessing, article-based chunking, retriever construction, context retrieval, and LLM-based answer generation.


### 3.1 Constructor

A RAG instance is created using the raw text and a specific configuration. In this example, custom preprocessing is enabled and article-based chunking is used, taking advantage of the legal structure of the document.


In [4]:
rag = RAG(
    raw_data=raw_text,
    use_custom_text_preprocessing=True,
    chunking={
        "chunking_method": "article_custom",
        "chunk_size_words": 250,
        "chunk_overlap_words": 50,
    },
    retriever_type="bm25",
    k=5,
    generation_model="gpt-4.1-mini",
)

rag.summary()


{'use_custom_text_preprocessing': True,
 'chunking': {'chunking_method': 'article_custom',
  'chunk_size_words': 250,
  'chunk_overlap_words': 50},
 'chunking_method': 'article_custom',
 'retriever_type': 'bm25',
 'embedding_model': 'text-embedding-3-small',
 'k': 5,
 'generation_model': 'gpt-4.1-mini',
 'clean_characters': 0,
 'articles': 0,
 'chunks': 0}

### 3.2 Text cleaning and preprocessing

In this step, the text is cleaned before being split into fragments. When custom preprocessing is enabled, formatting artifacts, headers, and unnecessary line breaks are removed to preserve a more useful version of the legal document.


In [5]:
clean_text = rag.preprocess_text()

print(f"Raw characters: {len(rag.raw_text):,}")
print(f"Clean characters: {len(clean_text):,}")
print(f"Main law characters: {len(rag.main_law_text):,}")
print(f"Transitory chunk words: {len(rag.transitory_chunk.split()):,}")

print(clean_text[:1_000])


Raw characters: 106,311
Clean characters: 103,630
Main law characters: 103,630
Transitory chunk words: 0
Reforma D.O.F. 09 de marzo 2018 Reforma D.O.F. 20 de mayo 2021 Reforma D.O.F. 16 de julio de 2025 SECRETARIA DE HACIENDA Y CREDITO PUBLICO DECRETO por el que se expide la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita. Al margen un sello con el Escudo Nacional, que dice: Estados Unidos Mexicanos. - Presidencia de la República. FELIPE DE JESÚS CALDERÓN HINOJOSA, Presidente de los Estados Unidos Mexicanos, a sus habitantes sabed: Que el Honorable Congreso de la Unión, se ha servido dirigirme el siguiente DECRETO "EL CONGRESO GENERAL DE LOS ESTADOS UNIDOS MEXICANOS, DECRETA: SE EXPIDE LA LEY FEDERAL PARA LA PREVENCIÓN E IDENTIFICACIÓN DE OPERACIONES CON RECURSOS DE PROCEDENCIA ILÍCITA. Artículo Único. - Se expide la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita: Ley Federal para la 

### 3.3 Chunking

Here, the preprocessed text is divided into chunks. The showcased configuration uses a custom article-based strategy with word size and overlap parameters, keeping relevant legal units together while avoiding overly long fragments.


In [6]:
chunks = rag.create_chunks()

print(f"Articles: {len(rag.articles):,}")
print(f"Chunks: {len(chunks):,}")
print(f"Chunk size: {rag.chunk_size_words:,} words")
print(f"Chunk overlap: {rag.chunk_overlap_words:,} words")

for chunk_index, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {chunk_index} | Words: {len(chunk.split()):,}")
    print(chunk[:1_000])


Articles: 5
Chunks: 82
Chunk size: 250 words
Chunk overlap: 50 words

Chunk 0 | Words: 250
Reforma D.O.F. 09 de marzo 2018 Reforma D.O.F. 20 de mayo 2021 Reforma D.O.F. 16 de julio de 2025 SECRETARIA DE HACIENDA Y CREDITO PUBLICO DECRETO por el que se expide la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita. Al margen un sello con el Escudo Nacional, que dice: Estados Unidos Mexicanos. - Presidencia de la República. FELIPE DE JESÚS CALDERÓN HINOJOSA, Presidente de los Estados Unidos Mexicanos, a sus habitantes sabed: Que el Honorable Congreso de la Unión, se ha servido dirigirme el siguiente DECRETO "EL CONGRESO GENERAL DE LOS ESTADOS UNIDOS MEXICANOS, DECRETA: SE EXPIDE LA LEY FEDERAL PARA LA PREVENCIÓN E IDENTIFICACIÓN DE OPERACIONES CON RECURSOS DE PROCEDENCIA ILÍCITA. Artículo Único. - Se expide la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita: Ley Federal para la Prevención e I

### 3.4 Retriever building

In this step, the retrieval mechanism is built over the generated chunks. The retriever makes it possible to compare a question against the document fragments and select the most relevant contexts.


In [7]:
retriever = rag.build_retriever()

print(f"Retriever type: {rag.retriever_type}")
print(f"Tokenized chunks: {len(rag.tokenized_chunks):,}")
print(f"Average tokens per chunk: {sum(len(tokens) for tokens in rag.tokenized_chunks) / len(rag.tokenized_chunks):,.1f}")


Retriever type: bm25
Tokenized chunks: 82
Average tokens per chunk: 248.7


### 3.5 Retrieval

Here, a search is executed with an example question. The system returns the highest-ranked chunks along with their ranking and score, making it possible to explicitly inspect the evidence that will be sent to the generation model.


In [8]:
question = homework_questions[0]
retrieved_chunks = rag.retrieve(question)

print(question)
print("\nChunks recuperados:")
for item in retrieved_chunks:
    print(
        f"Rank {item['rank']:>2} | "
        f"Chunk {item['chunk_index']:>3} | "
        f"Score {item['score']:>8.4f} | "
        f"Text: {item['chunk']}"
    )


¿Cuál es el objeto de la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita?

Chunks recuperados:
Rank  1 | Chunk   0 | Score  33.9918 | Text: Reforma D.O.F. 09 de marzo 2018 Reforma D.O.F. 20 de mayo 2021 Reforma D.O.F. 16 de julio de 2025 SECRETARIA DE HACIENDA Y CREDITO PUBLICO DECRETO por el que se expide la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita. Al margen un sello con el Escudo Nacional, que dice: Estados Unidos Mexicanos. - Presidencia de la República. FELIPE DE JESÚS CALDERÓN HINOJOSA, Presidente de los Estados Unidos Mexicanos, a sus habitantes sabed: Que el Honorable Congreso de la Unión, se ha servido dirigirme el siguiente DECRETO "EL CONGRESO GENERAL DE LOS ESTADOS UNIDOS MEXICANOS, DECRETA: SE EXPIDE LA LEY FEDERAL PARA LA PREVENCIÓN E IDENTIFICACIÓN DE OPERACIONES CON RECURSOS DE PROCEDENCIA ILÍCITA. Artículo Único. - Se expide la Ley Federal para la Prevención e I

### 3.6 Answer generation

In this step, an answer is generated using only the retrieved chunks as context. The prompt is designed to answer strictly and avoid using information that is not supported by the retrieved evidence.


In [9]:
rag_response = rag.generate_answer(
    question=question,
    client=client,
)

print(rag_response["answer"])


El objeto de la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita es proteger el sistema financiero y la economía nacional, estableciendo medidas y procedimientos para prevenir y detectar actos u operaciones que involucren recursos de procedencia ilícita, a través de una coordinación interinstitucional que recabe elementos útiles para investigar y perseguir dichos delitos, los relacionados con estos y las estructuras financieras de las organizaciones delictivas (Artículo 2).


## 4. EVALS Class showcase

This section demonstrates how the EVALS class works. This class automatically evaluates the RAG system responses across three dimensions: answer relevancy, faithfulness to the retrieved context, and context relevancy.


### 4.1 Constructor

The evaluation class is initialized with the model that will act as judge. This evaluator assigns qualitative labels and numerical values to compare RAG performance.


In [10]:
evals = EVALS(
    client=client,
    evaluation_model="gpt-4.1-mini",
)


### 4.2 Evaluation input

Here, answers are generated for multiple questions and stored together with the retrieved chunks. These results are the input used by the following evaluation steps.


In [11]:
rag_results_showcase = []

for question_number, question in enumerate(homework_questions, start=1):
    response = rag.generate_answer(
        question=question,
        client=client,
    )
    rag_results_showcase.append({
        "question_number": question_number,
        "question": response["question"],
        "answer": response["answer"],
        "retrieved_chunks": response["retrieved_chunks"],
        "retrieved_context": response["retrieved_context"],
    })
    print("\n" + "-" * 100)
    print(f"Pregunta {question_number}:")
    print(question)
    print("\nRespuesta:")
    print(response["answer"])
    print("\nChunks recuperados:")
    for item in response["retrieved_chunks"]:
        print(
            f"Rank {item['rank']:>2} | "
            f"Chunk {item['chunk_index']:>3} | "
            f"Score {item['score']:>8.4f} | "
            f"Text: {item['chunk']}"
        )

print(f"\nTotal RAG results: {len(rag_results_showcase)}")



----------------------------------------------------------------------------------------------------
Pregunta 1:
¿Cuál es el objeto de la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita?

Respuesta:
El objeto de la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita es proteger el sistema financiero y la economía nacional, estableciendo medidas y procedimientos para prevenir y detectar actos u operaciones que involucren recursos de procedencia ilícita, mediante una coordinación interinstitucional para recabar elementos útiles para investigar y perseguir los delitos relacionados (Artículo 2).

Chunks recuperados:
Rank  1 | Chunk   0 | Score  33.9918 | Text: Reforma D.O.F. 09 de marzo 2018 Reforma D.O.F. 20 de mayo 2021 Reforma D.O.F. 16 de julio de 2025 SECRETARIA DE HACIENDA Y CREDITO PUBLICO DECRETO por el que se expide la Ley Federal para la Prevención e Identificación de Operaciones co

### 4.3 Answer relevancy

This evaluation measures whether the answer directly addresses the question. It does not verify by itself whether the answer is true; it only evaluates whether the answer is relevant to the question.


In [12]:
answer_relevancy_df = evals.evaluate_answer_relevancy(rag_results_showcase)
answer_relevancy_df


,question_number,question,answer,answer_relevancy_label,answer_relevancy_score,answer_relevancy_reasoning
0,1,¿Cuál es el objeto de la Ley Federal para la P...,El objeto de la Ley Federal para la Prevención...,GOOD,2,La respuesta explica claramente el objeto de l...
1,2,¿Qué autoridades llevarán a cabo la supervisió...,"La supervisión, verificación y vigilancia del ...",GOOD,2,La respuesta identifica claramente las autorid...
2,3,¿A partir de qué monto la comercialización o d...,La comercialización o distribución habitual o ...,GOOD,2,La respuesta especifica claramente el monto a ...
3,4,¿Cuál es el plazo para presentar un Aviso en c...,El plazo para presentar un Aviso en caso de so...,GOOD,2,La respuesta indica claramente el plazo especí...
4,5,¿Cuáles son las sanciones por proporcionar inf...,Las sanciones por proporcionar información o d...,GOOD,2,La respuesta responde de forma directa y clara...


### 4.4 Faithfulness

This evaluation measures whether the answer is supported by the retrieved context. An answer can still be considered faithful when it says there is not enough information, as long as that lack of information is consistent with the available context.


In [13]:
faithfulness_df = evals.evaluate_faithfulness(rag_results_showcase)
faithfulness_df


,question_number,question,answer,faithfulness_label,faithfulness_score,faithfulness_reasoning
0,1,¿Cuál es el objeto de la Ley Federal para la P...,El objeto de la Ley Federal para la Prevención...,GOOD,2,La respuesta reproduce fielmente el contenido ...
1,2,¿Qué autoridades llevarán a cabo la supervisió...,"La supervisión, verificación y vigilancia del ...",GOOD,2,La respuesta se basa directamente en el Artícu...
2,3,¿A partir de qué monto la comercialización o d...,La comercialización o distribución habitual o ...,GOOD,2,La respuesta cita textualmente el contexto rec...
3,4,¿Cuál es el plazo para presentar un Aviso en c...,El plazo para presentar un Aviso en caso de so...,GOOD,2,La respuesta está directamente sustentada en e...
4,5,¿Cuáles son las sanciones por proporcionar inf...,Las sanciones por proporcionar información o d...,GOOD,2,La respuesta se basa directamente en el Artícu...


### 4.5 Context relevancy

This evaluation measures whether the complete set of retrieved chunks contains useful information to answer the question. The context is evaluated as a whole, not chunk by chunk.


In [14]:
context_relevancy_df = evals.evaluate_context_relevancy(rag_results_showcase)
context_relevancy_df


,question_number,question,retrieved_chunk_indexes,context_relevancy_label,context_relevancy_score,context_relevancy_reasoning
0,1,¿Cuál es el objeto de la Ley Federal para la P...,"[0, 75, 78, 80, 9]",GOOD,2,El contexto recuperado incluye explícitamente ...
1,2,¿Qué autoridades llevarán a cabo la supervisió...,"[17, 40, 16, 62, 51]",GOOD,2,El contexto recuperado menciona explícitamente...
2,3,¿A partir de qué monto la comercialización o d...,"[22, 21, 33, 19, 31]",GOOD,2,El contexto recuperado especifica claramente q...
3,4,¿Cuál es el plazo para presentar un Aviso en c...,"[35, 34, 0, 11, 10]",GOOD,2,El contexto recuperado especifica claramente q...
4,5,¿Cuáles son las sanciones por proporcionar inf...,"[66, 69, 59, 51, 57]",GOOD,2,El contexto incluye el Artículo 62 que estable...


### 4.6 All evals

Here, all evaluations are executed together and a summary is generated with the average score per evaluation type and the overall performance value.


In [15]:
eval_results_showcase = evals.evaluate_all(rag_results_showcase)
summary_df = eval_results_showcase["summary"]
combined_df = eval_results_showcase["combined"]

display(combined_df)
display(summary_df)


,question_number,question,answer,answer_relevancy_label,answer_relevancy_score,answer_relevancy_reasoning,faithfulness_label,faithfulness_score,faithfulness_reasoning,retrieved_chunk_indexes,context_relevancy_label,context_relevancy_score,context_relevancy_reasoning
0,1,¿Cuál es el objeto de la Ley Federal para la P...,El objeto de la Ley Federal para la Prevención...,GOOD,2,La respuesta explica claramente el objeto de l...,GOOD,2,La respuesta reproduce fielmente el contenido ...,"[0, 75, 78, 80, 9]",GOOD,2,El contexto recuperado incluye explícitamente ...
1,2,¿Qué autoridades llevarán a cabo la supervisió...,"La supervisión, verificación y vigilancia del ...",GOOD,2,La respuesta identifica claramente las autorid...,GOOD,2,La respuesta se basa directamente en el Artícu...,"[17, 40, 16, 62, 51]",GOOD,2,El contexto recuperado menciona explícitamente...
2,3,¿A partir de qué monto la comercialización o d...,La comercialización o distribución habitual o ...,GOOD,2,La respuesta especifica claramente el monto a ...,GOOD,2,La respuesta coincide exactamente con la infor...,"[22, 21, 33, 19, 31]",GOOD,2,El contexto recuperado especifica claramente q...
3,4,¿Cuál es el plazo para presentar un Aviso en c...,El plazo para presentar un Aviso en caso de so...,GOOD,2,La respuesta indica claramente el plazo especí...,GOOD,2,La respuesta está directamente sustentada en e...,"[35, 34, 0, 11, 10]",GOOD,2,El contexto recuperado especifica claramente q...
4,5,¿Cuáles son las sanciones por proporcionar inf...,Las sanciones por proporcionar información o d...,GOOD,2,La respuesta responde de forma directa y clara...,GOOD,2,La respuesta se basa directamente en el Artícu...,"[66, 69, 59, 51, 57]",GOOD,2,El contexto incluye el Artículo 62 que estable...


,eval_type,average_score,normalized_score
0,answer_relevancy,2.0,1.0
1,faithfulness,2.0,1.0
2,context_relevancy,2.0,1.0
3,overall,2.0,1.0


## 5. RAG configuration evaluation

This section compares multiple RAG system configurations. Parameters such as chunking strategy, custom preprocessing, retriever type, number of retrieved chunks, and embedding model are varied.


### 5.1 Configuration list

Here, the configuration search space is defined. The configurations combine different strategies, but article-based custom chunking is not used without custom preprocessing because that strategy depends on the cleaned legal format.


In [16]:
base_rag_configurations = [
    {
        "config_name": "article_custom_k3",
        "chunking": {
            "chunking_method": "article_custom",
            "chunk_size_words": 400,
            "chunk_overlap_words": 80,
        },
        "k": 3,
        "generation_model": "gpt-5-nano",
    },
    {
        "config_name": "article_custom_k5",
        "chunking": {
            "chunking_method": "article_custom",
            "chunk_size_words": 400,
            "chunk_overlap_words": 80,
        },
        "k": 5,
        "generation_model": "gpt-5-nano",
    },
    {
        "config_name": "size_overlap_250_50_k5",
        "chunking": {
            "chunking_method": "size_overlap",
            "chunk_size_words": 250,
            "chunk_overlap_words": 50,
        },
        "k": 5,
        "generation_model": "gpt-5-nano",
    },
    {
        "config_name": "size_overlap_400_80_k5",
        "chunking": {
            "chunking_method": "size_overlap",
            "chunk_size_words": 400,
            "chunk_overlap_words": 80,
        },
        "k": 5,
        "generation_model": "gpt-5-nano",
    },
    {
        "config_name": "semantic_040_k5",
        "chunking": {
            "chunking_method": "semantic",
            "similarity_threshold": 0.4,
            "sentence_window_size": 1,
        },
        "k": 5,
        "generation_model": "gpt-5-nano",
    },
]

retriever_configurations = [
    {
        "retriever_type": "bm25",
    },
    {
        "retriever_type": "openai_embeddings",
        "embedding_model": "text-embedding-3-small",
    },
]

rag_configurations = []

for base_config in base_rag_configurations:
    chunking_method = base_config["chunking"]["chunking_method"]
    preprocessing_options = [True] if chunking_method == "article_custom" else [True, False]

    for retriever_config in retriever_configurations:
        for use_custom_text_preprocessing in preprocessing_options:
            preprocessing_suffix = "custom_preprocessing" if use_custom_text_preprocessing else "simple_preprocessing"

            config = {
                **base_config,
                **retriever_config,
                "config_name": (
                    f"{retriever_config['retriever_type']}_"
                    f"{base_config['config_name']}_"
                    f"{preprocessing_suffix}"
                ),
                "use_custom_text_preprocessing": use_custom_text_preprocessing,
            }
            rag_configurations.append(config)

pd.DataFrame(rag_configurations)


,config_name,chunking,k,generation_model,retriever_type,use_custom_text_preprocessing,embedding_model
0,bm25_article_custom_k3_custom_preprocessing,"{'chunking_method': 'article_custom', 'chunk_s...",3,gpt-5-nano,bm25,True,NaN
1,openai_embeddings_article_custom_k3_custom_pre...,"{'chunking_method': 'article_custom', 'chunk_s...",3,gpt-5-nano,openai_embeddings,True,text-embedding-3-small
2,bm25_article_custom_k5_custom_preprocessing,"{'chunking_method': 'article_custom', 'chunk_s...",5,gpt-5-nano,bm25,True,NaN
3,openai_embeddings_article_custom_k5_custom_pre...,"{'chunking_method': 'article_custom', 'chunk_s...",5,gpt-5-nano,openai_embeddings,True,text-embedding-3-small
4,bm25_size_overlap_250_50_k5_custom_preprocessing,"{'chunking_method': 'size_overlap', 'chunk_siz...",5,gpt-5-nano,bm25,True,NaN
5,bm25_size_overlap_250_50_k5_simple_preprocessing,"{'chunking_method': 'size_overlap', 'chunk_siz...",5,gpt-5-nano,bm25,False,NaN
6,openai_embeddings_size_overlap_250_50_k5_custo...,"{'chunking_method': 'size_overlap', 'chunk_siz...",5,gpt-5-nano,openai_embeddings,True,text-embedding-3-small
7,openai_embeddings_size_overlap_250_50_k5_simpl...,"{'chunking_method': 'size_overlap', 'chunk_siz...",5,gpt-5-nano,openai_embeddings,False,text-embedding-3-small
8,bm25_size_overlap_400_80_k5_custom_preprocessing,"{'chunking_method': 'size_overlap', 'chunk_siz...",5,gpt-5-nano,bm25,True,NaN
9,bm25_size_overlap_400_80_k5_simple_preprocessing,"{'chunking_method': 'size_overlap', 'chunk_siz...",5,gpt-5-nano,bm25,False,NaN


### 5.2 Experiment runner

This function encapsulates the experimental workflow for one configuration: creating the RAG object, preprocessing the text, creating chunks, building the retriever, retrieving context, and generating answers for all questions.


In [17]:
def run_rag_configuration(config, questions, raw_text, client):
    rag = RAG(
        raw_data=raw_text,
        use_custom_text_preprocessing=config["use_custom_text_preprocessing"],
        chunking=config["chunking"],
        retriever_type=config["retriever_type"],
        embedding_model=config.get("embedding_model", "text-embedding-3-small"),
        k=config["k"],
        generation_model=config["generation_model"],
    )

    rag.preprocess_text()
    rag.create_chunks(client=client)
    rag.build_retriever(client=client)

    rag_results = []
    for question_number, question in enumerate(questions, start=1):
        response = rag.generate_answer(
            question=question,
            client=client,
        )
        rag_results.append({
            "config_name": config["config_name"],
            "question_number": question_number,
            "question": response["question"],
            "answer": response["answer"],
            "retrieved_chunks": response["retrieved_chunks"],
            "retrieved_context": response["retrieved_context"],
            "rag_summary": rag.summary(),
        })

    return rag_results


### 5.3 Run configurations

Here, all defined configurations are executed over the homework questions. For each configuration, answers are generated, evaluated with the EVALS class, and stored in a comparable summary.


In [18]:
all_rag_results = []
all_eval_results = {}
configuration_summaries = []

for config in rag_configurations:
    print("\n" + "-" * 100)
    print(f"Running configuration: {config['config_name']}")

    config_rag_results = run_rag_configuration(
        config=config,
        questions=homework_questions,
        raw_text=raw_text,
        client=client,
    )

    config_eval_results = evals.evaluate_all(config_rag_results)
    config_summary = config_eval_results["summary"].copy()
    config_summary.insert(0, "config_name", config["config_name"])

    all_rag_results.extend(config_rag_results)
    all_eval_results[config["config_name"]] = config_eval_results
    configuration_summaries.append(config_summary)

    overall_score = config_summary.loc[
        config_summary["eval_type"] == "overall",
        "average_score",
    ].iloc[0]

    print(f"Overall score: {overall_score:.4f}")

configuration_summary_df = pd.concat(configuration_summaries, ignore_index=True)
configuration_summary_df



----------------------------------------------------------------------------------------------------
Running configuration: bm25_article_custom_k3_custom_preprocessing
Overall score: 2.0000

----------------------------------------------------------------------------------------------------
Running configuration: openai_embeddings_article_custom_k3_custom_preprocessing
Overall score: 1.8000

----------------------------------------------------------------------------------------------------
Running configuration: bm25_article_custom_k5_custom_preprocessing
Overall score: 2.0000

----------------------------------------------------------------------------------------------------
Running configuration: openai_embeddings_article_custom_k5_custom_preprocessing
Overall score: 2.0000

----------------------------------------------------------------------------------------------------
Running configuration: bm25_size_overlap_250_50_k5_custom_preprocessing
Overall score: 2.0000

-------------

,config_name,eval_type,average_score,normalized_score
0,bm25_article_custom_k3_custom_preprocessing,answer_relevancy,2.000000,1.000000
1,bm25_article_custom_k3_custom_preprocessing,faithfulness,2.000000,1.000000
2,bm25_article_custom_k3_custom_preprocessing,context_relevancy,2.000000,1.000000
3,bm25_article_custom_k3_custom_preprocessing,overall,2.000000,1.000000
4,openai_embeddings_article_custom_k3_custom_pre...,answer_relevancy,1.800000,0.900000
...,...,...,...,...
59,openai_embeddings_semantic_040_k5_custom_prepr...,overall,1.866667,0.933333
60,openai_embeddings_semantic_040_k5_simple_prepr...,answer_relevancy,2.000000,1.000000
61,openai_embeddings_semantic_040_k5_simple_prepr...,faithfulness,2.000000,1.000000
62,openai_embeddings_semantic_040_k5_simple_prepr...,context_relevancy,2.000000,1.000000


### 5.4 Configuration comparison

This subsection organizes the evaluation results into a comparison table. Configurations are sorted by their overall value to identify which ones performed best under the defined evaluation scheme.


In [19]:
configuration_comparison_df = configuration_summary_df.pivot(
    index="config_name",
    columns="eval_type",
    values="average_score",
).reset_index()

configuration_comparison_df = configuration_comparison_df.sort_values(
    by="overall",
    ascending=False,
).reset_index(drop=True)

configuration_comparison_df


eval_type,config_name,answer_relevancy,context_relevancy,faithfulness,overall
0,bm25_article_custom_k3_custom_preprocessing,2.0,2.0,2.0,2.000000
1,bm25_article_custom_k5_custom_preprocessing,2.0,2.0,2.0,2.000000
2,bm25_semantic_040_k5_simple_preprocessing,2.0,2.0,2.0,2.000000
3,bm25_size_overlap_250_50_k5_custom_preprocessing,2.0,2.0,2.0,2.000000
4,bm25_size_overlap_250_50_k5_simple_preprocessing,2.0,2.0,2.0,2.000000
5,bm25_size_overlap_400_80_k5_custom_preprocessing,2.0,2.0,2.0,2.000000
6,bm25_size_overlap_400_80_k5_simple_preprocessing,2.0,2.0,2.0,2.000000
7,openai_embeddings_article_custom_k5_custom_pre...,2.0,2.0,2.0,2.000000
8,openai_embeddings_semantic_040_k5_simple_prepr...,2.0,2.0,2.0,2.000000
9,openai_embeddings_size_overlap_250_50_k5_custo...,2.0,2.0,2.0,2.000000


### 5.5 Best configuration

Here, the configurations with the best overall value are identified. Since several configurations may obtain the same maximum score, all ties are preserved before selecting a final configuration.


In [20]:
best_overall_score = configuration_comparison_df["overall"].max()
best_configurations_df = configuration_comparison_df[
    configuration_comparison_df["overall"] == best_overall_score
].reset_index(drop=True)

best_configuration_names = best_configurations_df["config_name"].tolist()
selected_best_configuration_name = "openai_embeddings_article_custom_k5_custom_preprocessing"

print(f"Best overall score: {best_overall_score:.4f}")
print("Best configuration(s) by evaluation score:")
for config_name in best_configuration_names:
    print(f"- {config_name}")

print(f"\nSelected configuration for homework answers: {selected_best_configuration_name}")
best_configurations_df


Best overall score: 2.0000
Best configuration(s) by evaluation score:
- bm25_article_custom_k3_custom_preprocessing
- bm25_article_custom_k5_custom_preprocessing
- bm25_semantic_040_k5_simple_preprocessing
- bm25_size_overlap_250_50_k5_custom_preprocessing
- bm25_size_overlap_250_50_k5_simple_preprocessing
- bm25_size_overlap_400_80_k5_custom_preprocessing
- bm25_size_overlap_400_80_k5_simple_preprocessing
- openai_embeddings_article_custom_k5_custom_preprocessing
- openai_embeddings_semantic_040_k5_simple_preprocessing
- openai_embeddings_size_overlap_250_50_k5_custom_preprocessing
- openai_embeddings_size_overlap_250_50_k5_simple_preprocessing
- openai_embeddings_size_overlap_400_80_k5_custom_preprocessing
- openai_embeddings_size_overlap_400_80_k5_simple_preprocessing

Selected configuration for homework answers: openai_embeddings_article_custom_k5_custom_preprocessing


eval_type,config_name,answer_relevancy,context_relevancy,faithfulness,overall
0,bm25_article_custom_k3_custom_preprocessing,2.0,2.0,2.0,2.0
1,bm25_article_custom_k5_custom_preprocessing,2.0,2.0,2.0,2.0
2,bm25_semantic_040_k5_simple_preprocessing,2.0,2.0,2.0,2.0
3,bm25_size_overlap_250_50_k5_custom_preprocessing,2.0,2.0,2.0,2.0
4,bm25_size_overlap_250_50_k5_simple_preprocessing,2.0,2.0,2.0,2.0
5,bm25_size_overlap_400_80_k5_custom_preprocessing,2.0,2.0,2.0,2.0
6,bm25_size_overlap_400_80_k5_simple_preprocessing,2.0,2.0,2.0,2.0
7,openai_embeddings_article_custom_k5_custom_pre...,2.0,2.0,2.0,2.0
8,openai_embeddings_semantic_040_k5_simple_prepr...,2.0,2.0,2.0,2.0
9,openai_embeddings_size_overlap_250_50_k5_custo...,2.0,2.0,2.0,2.0


### 5.6 Best configuration details

This section shows the detailed evaluation results for the configurations tied with the best score. This makes it possible to review how the highest-scoring models performed question by question.


In [21]:
best_configuration_results = []

for config_name in best_configuration_names:
    combined_df = all_eval_results[config_name]["combined"].copy()
    combined_df.insert(0, "config_name", config_name)
    best_configuration_results.append(combined_df)

best_configuration_results_df = pd.concat(best_configuration_results, ignore_index=True)
best_configuration_results_df


,config_name,question_number,question,answer,answer_relevancy_label,answer_relevancy_score,answer_relevancy_reasoning,faithfulness_label,faithfulness_score,faithfulness_reasoning,retrieved_chunk_indexes,context_relevancy_label,context_relevancy_score,context_relevancy_reasoning
0,bm25_article_custom_k3_custom_preprocessing,1,¿Cuál es el objeto de la Ley Federal para la P...,El objeto es proteger el sistema financiero y ...,GOOD,2,La respuesta explica claramente el objeto de l...,GOOD,2,La respuesta reproduce fielmente el contenido ...,"[0, 2, 48]",GOOD,2,El contexto recuperado incluye el artículo 2 d...
1,bm25_article_custom_k3_custom_preprocessing,2,¿Qué autoridades llevarán a cabo la supervisió...,"Según el Artículo 16, la supervisión, verifica...",GOOD,2,La respuesta identifica claramente las autorid...,GOOD,2,La respuesta se basa directamente en el Artícu...,"[10, 25, 7]",GOOD,2,El contexto recuperado incluye explícitamente ...
2,bm25_article_custom_k3_custom_preprocessing,3,¿A partir de qué monto la comercialización o d...,Se considera Actividad Vulnerable sujeta a Avi...,GOOD,2,La respuesta especifica claramente el monto a ...,GOOD,2,La respuesta indica correctamente que la comer...,"[13, 14, 18]",GOOD,2,El contexto recuperado especifica claramente q...
3,bm25_article_custom_k3_custom_preprocessing,4,¿Cuál es el plazo para presentar un Aviso en c...,- Plazo: Presentar el Aviso dentro de las 24 h...,GOOD,2,La respuesta indica claramente el plazo para p...,GOOD,2,La respuesta indica correctamente que el Aviso...,"[21, 22, 0]",GOOD,2,El contexto recuperado especifica claramente q...
4,bm25_article_custom_k3_custom_preprocessing,5,¿Cuáles son las sanciones por proporcionar inf...,"Artículo 62, fracción I: Se sancionará con pri...",GOOD,2,La respuesta proporciona directamente las sanc...,GOOD,2,"La respuesta cita textualmente el Artículo 62,...","[42, 36, 44]",GOOD,2,El contexto incluye el Artículo 62 que estable...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,openai_embeddings_size_overlap_400_80_k5_simpl...,1,¿Cuál es el objeto de la Ley Federal para la P...,El objeto de la Ley es proteger el sistema fin...,GOOD,2,La respuesta explica claramente el objeto de l...,GOOD,2,La respuesta reproduce textualmente el objeto ...,"[0, 5, 50, 7, 36]",GOOD,2,El contexto incluye el Artículo 2 de la Ley Fe...
61,openai_embeddings_size_overlap_400_80_k5_simpl...,2,¿Qué autoridades llevarán a cabo la supervisió...,Serán la Comisión Nacional Bancaria y de Valor...,GOOD,2,La respuesta identifica claramente las autorid...,GOOD,2,La respuesta está directamente sustentada en e...,"[10, 9, 11, 40, 8]",GOOD,2,El contexto recuperado incluye explícitamente ...
62,openai_embeddings_size_overlap_400_80_k5_simpl...,3,¿A partir de qué monto la comercialización o d...,A partir de un monto igual o superior a seis m...,GOOD,2,La respuesta indica claramente el monto a part...,GOOD,2,La respuesta indica correctamente que la comer...,"[14, 27, 13, 11, 26]",GOOD,2,El contexto recuperado especifica claramente q...
63,openai_embeddings_size_overlap_400_80_k5_simpl...,4,¿Cuál es el plazo para presentar un Aviso en c...,A más tardar el día 17 del mes inmediato sigui...,GOOD,2,La respuesta indica claramente el plazo especí...,GOOD,2,La respuesta indica correctamente que el plazo...,"[41, 35, 34, 7, 26]",GOOD,2,El contexto recuperado incluye el artículo 23 ...


## 6. Homework questions with best RAG

This section generates the final answers for the homework questions using a configuration selected from the previous analysis. Although several configurations may be tied as best-performing models, one concrete configuration is used to produce the final answers consistently.


### 6.1 Best RAG configuration

Here, the `openai_embeddings_article_custom_k5_custom_preprocessing` configuration is selected. This selection is arbitrary among the strong-performing configurations, but it was chosen because it uses the custom features developed for this problem: legal-text preprocessing, article-based chunking, and OpenAI embeddings.


In [ ]:
best_configuration_name = selected_best_configuration_name
best_configuration = next(
    config for config in rag_configurations
    if config["config_name"] == best_configuration_name
)

selected_configuration_eval_df = configuration_comparison_df[
    configuration_comparison_df["config_name"] == best_configuration_name
].reset_index(drop=True)

print(f"Selected configuration: {best_configuration_name}")
print(f"Number of tied best configurations by evaluation score: {len(best_configuration_names)}")

if len(selected_configuration_eval_df) > 0:
    display(selected_configuration_eval_df)

best_configuration


Selected configuration: openai_embeddings_article_custom_k5_custom_preprocessing
Number of tied best configurations by evaluation score: 13


eval_type,config_name,answer_relevancy,context_relevancy,faithfulness,overall
0,openai_embeddings_article_custom_k5_custom_pre...,2.0,2.0,2.0,2.0


{'config_name': 'openai_embeddings_article_custom_k5_custom_preprocessing',
 'chunking': {'chunking_method': 'article_custom',
  'chunk_size_words': 400,
  'chunk_overlap_words': 80},
 'k': 5,
 'generation_model': 'gpt-5-nano',
 'retriever_type': 'openai_embeddings',
 'embedding_model': 'text-embedding-3-small',
 'use_custom_text_preprocessing': True}

### 6.2 Homework answers

Finally, the selected configuration is executed over the homework questions. The generated answers and retrieved chunks are printed so both the final answer and the supporting evidence used by the system can be reviewed.


In [23]:
homework_rag_results = run_rag_configuration(
    config=best_configuration,
    questions=homework_questions,
    raw_text=raw_text,
    client=client,
)

homework_answers_df = pd.DataFrame([
    {
        "question_number": result["question_number"],
        "question": result["question"],
        "answer": result["answer"],
        "retrieved_chunk_indexes": [
            item["chunk_index"] for item in result["retrieved_chunks"]
        ],
    }
    for result in homework_rag_results
])

for result in homework_rag_results:
    print("\n" + "=" * 100)
    print(f"Pregunta {result['question_number']}:")
    print(result["question"])
    print("\nRespuesta:")
    print(result["answer"])
    print("\nChunks recuperados:")
    for item in result["retrieved_chunks"]:
        print(
            f"Rank {item['rank']:>2} | "
            f"Chunk {item['chunk_index']:>3} | "
            f"Score {item['score']:>8.4f} | "
            f"Text: {item['chunk']}"
        )

display(homework_answers_df)



Pregunta 1:
¿Cuál es el objeto de la Ley Federal para la Prevención e Identificación de Operaciones con Recursos de Procedencia Ilícita?

Respuesta:
El objeto es proteger el sistema financiero y la economía nacional, estableciendo medidas y procedimientos para prevenir y detectar actos u operaciones que involucren recursos de procedencia ilícita, a través de una coordinación interinstitucional, con fines de recabar elementos útiles para investigar y perseguir los Delitos de Operaciones con Recursos de Procedencia Ilícita, los delitos relacionados y las estructuras financieras de las organizaciones delictivas, así como evitar el uso de los recursos para su financiamiento. (Artículo 2)

Chunks recuperados:
Rank  1 | Chunk   5 | Score   0.7160 | Text: cuando, con motivo del ejercicio de sus atribuciones, identifique hechos que puedan constituir delitos; V. Requerir la comparecencia de presuntos infractores y demás personas que puedan contribuir a la verificación del cumplimiento de las o

,question_number,question,answer,retrieved_chunk_indexes
0,1,¿Cuál es el objeto de la Ley Federal para la P...,El objeto es proteger el sistema financiero y ...,"[5, 0, 50, 35, 3]"
1,2,¿Qué autoridades llevarán a cabo la supervisió...,Será llevada a cabo por la Comisión Nacional B...,"[10, 9, 39, 4, 6]"
2,3,¿A partir de qué monto la comercialización o d...,A partir de un monto igual o superior al equiv...,"[14, 13, 18, 19, 11]"
3,4,¿Cuál es el plazo para presentar un Aviso en c...,Dentro de las 24 horas siguientes a que tuvier...,"[22, 33, 34, 8, 21]"
4,5,¿Cuáles son las sanciones por proporcionar inf...,"Según el Artículo 62, fracción I, se sancionar...","[33, 40, 45, 39, 44]"
